In [1]:
# Install PyTorch
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install Diffusers and dependencies
!pip install diffusers transformers accelerate
!pip install opencv-python pillow numpy matplotlib
!pip install controlnet-aux xformers invisible-watermark


Looking in indexes: https://download.pytorch.org/whl/cu118

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
  Using cached xformers-0.0.31.post1.tar.gz (12.1 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached invisible_watermark-0.2.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached torch-2.7.1-cp310-none-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached torchvision-0.22.1-cp310-cp310-maco

In [ ]:
# import subprocess
# import sys
# import os

# packages = [
#     "torch", "torchvision", "torchaudio",
#     "transformers", "diffusers", "controlnet-aux", 
#     "opencv-python", "pillow", "numpy", "matplotlib",
#     "accelerate", "xformers", "compel"
# ]

# for package in packages:
#     try:
#         subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])
#     except:
#         pass

print("Dependencies installed successfully!")

# Now let's load and prepare the images
import torch
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from diffusers import (
    StableDiffusionInpaintPipeline, 
    StableDiffusionControlNetPipeline,
    StableDiffusionXLInpaintPipeline,
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    StableDiffusionPipeline,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler
)
from controlnet_aux import CannyDetector, OpenposeDetector#, DepthEstimator
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"

Dependencies installed successfully!


In [7]:
src_img = Image.open("../frames/glasses.png").convert("RGB").resize((512, 512))
mask_img = Image.open("../frames/glasses_mask.png").convert("L").resize((512, 512))
bg_img = Image.open("../frames/facetest2.png").convert("RGB").resize((512, 512))

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def method1_sd_inpainting(glasses_img, mask_img, face_img, device="cuda"):
    """
    SD 1.5 Inpainting Harmonization
    Uses masked inpainting with text conditioning for harmonization
    """
    from diffusers import StableDiffusionInpaintPipeline
    
    # Load inpainting pipeline
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        torch_dtype=torch.float16,
        use_safetensors=True
    ).to(device)
    
    # Prepare images
    face_resized = face_img.resize((512, 512))
    mask_resized = mask_img.resize((512, 512))
    
    # Invert mask (white = inpaint, black = keep)
    mask_array = np.array(mask_resized)
    mask_inverted = Image.fromarray(255 - mask_array)
    
    # Generate harmonized image
    result = pipe(
        prompt="realistic face with glasses, natural lighting, photorealistic, high quality",
        image=face_resized,
        mask_image=mask_inverted,
        negative_prompt="blurry, low quality, distorted, unrealistic",
        num_inference_steps=20,
        strength=0.8,
        guidance_scale=7.5
    ).images[0]
    
    return result

result = method1_sd_inpainting(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

OSError: Could not find the necessary `safetensors` weights in {'README.md', 'vae/diffusion_pytorch_model.bin', '.gitattributes', 'safety_checker/pytorch_model.bin', 'text_encoder/pytorch_model.bin', 'model_index.json', 'unet/config.json', 'safety_checker/pytorch_model.fp16.bin', 'safety_checker/config.json', 'tokenizer/vocab.json', 'vae/diffusion_pytorch_model.fp16.bin', 'feature_extractor/preprocessor_config.json', 'text_encoder/pytorch_model.fp16.bin', 'tokenizer/merges.txt', 'vae/config.json', 'unet/diffusion_pytorch_model.fp16.bin', 'safety_checker/model.fp16.safetensors', 'sd-v1-5-inpainting.ckpt', 'unet/diffusion_pytorch_model.bin', 'scheduler/scheduler_config.json', 'vae/diffusion_pytorch_model.fp16.safetensors', 'tokenizer/special_tokens_map.json', 'tokenizer/tokenizer_config.json', 'text_encoder/config.json', 'config.json', 'text_encoder/model.fp16.safetensors', 'unet/diffusion_pytorch_model.fp16.safetensors'} (variant=None)

In [ ]:
from diffusers import StableDiffusionXLInpaintPipeline

# Load SDXL inpainting pipeline
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    torch_dtype=torch.float16,
    use_safetensors=True
).to(device)

def method2_sdxl_inpainting(glasses_img, mask_img, face_img, device="cuda"):
    """
    SDXL Inpainting Harmonization
    Uses SDXL inpainting model for high-quality harmonization
    """
    
    # Prepare for SDXL (1024x1024)
    face_resized = face_img.resize((1024, 1024))
    mask_resized = mask_img.resize((1024, 1024))
    mask_inverted = Image.fromarray(255 - np.array(mask_resized))
    
    # Generate with SDXL
    result = pipe(
        prompt="person wearing glasses, professional photography, natural lighting, detailed face",
        image=face_resized,
        mask_image=mask_inverted,
        negative_prompt="blurry, low quality, distorted, artificial",
        num_inference_steps=30,
        strength=0.7,
        guidance_scale=8.0
    ).images[0]
    
    return result.resize((512, 512))



Fetching 18 files:  17%|█▋        | 3/18 [00:40<03:21, 13.44s/it]


KeyboardInterrupt: 

In [ ]:
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from controlnet_aux import CannyDetector

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load ControlNet model
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0",
    torch_dtype=dtype,
    use_safetensors=True
)

# Load SDXL pipeline with ControlNet
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=dtype,
    use_safetensors=True
).to(device)

canny_detector = CannyDetector()


def method3_sdxl_controlnet_canny(glasses_img, mask_img, face_img, device="cuda"):
    """
    MANDATORY: SDXL + ControlNet Canny + Mask Harmonization
    Uses Canny edge detection from glasses with mask conditioning
    """
    
    # Prepare images
    face_resized = face_img.resize((1024, 1024))
    glasses_resized = glasses_img.resize((512, 512))
    mask_resized = mask_img.resize((1024, 1024))
    
    # Generate Canny edges from glasses
    canny_image = canny_detector(glasses_resized)
    
    # Generate with ControlNet conditioning
    with torch.inference_mode():
        result = pipe(
            prompt="person wearing glasses, natural lighting, photorealistic, detailed face",
            image=canny_image,
            negative_prompt="blurry, low quality, distorted, unrealistic",
            num_inference_steps=5,
            controlnet_conditioning_scale=0.8,
            guidance_scale=7.5
        ).images[0]
    
    # Apply mask blending
    result_np = np.array(result.resize((512, 512)))
    face_np = np.array(face_img)
    mask_np = np.array(mask_img)
    mask_3d = np.stack([mask_np/255.0] * 3, axis=-1)
    
    final_result = face_np * (1 - mask_3d) + result_np * mask_3d
    return Image.fromarray(final_result.astype(np.uint8))

result = method3_sdxl_controlnet_canny(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

Loading pipeline components...:  43%|████▎     | 3/7 [00:00<00:00, 17.60it/s]

In [ ]:
# ===================================================================
# METHOD 4: CONTROLNET DEPTH HARMONIZATION
# ===================================================================

def method4_controlnet_depth(glasses_img, mask_img, face_img, device="cpu"):
    """
    Method 4: ControlNet Depth Harmonization
    Uses depth estimation for 3D-aware harmonization
    """
    
    print("Loading ControlNet Depth model...")
    
    try:
        # Load ControlNet depth model
        controlnet_depth = ControlNetModel.from_pretrained(
            "diffusers/controlnet-depth-sdxl-1.0",
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            use_safetensors=True
        )
        
        # Load pipeline
        pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            controlnet=controlnet_depth,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            variant="fp16" if device == "cuda" else None,
            use_safetensors=True
        )
        
        pipe = pipe.to(device)
        
        # Create depth map (simplified)
        face_resized = face_img.resize((1024, 1024))
        face_gray = face_resized.convert('L')
        
        # Simple depth estimation (fallback)
        depth_map = face_gray.filter(ImageFilter.GaussianBlur(radius=2))
        depth_map = depth_map.convert('RGB')
        
        # Generate harmonized image
        prompt = "person with glasses, natural depth, realistic lighting, detailed facial features"
        negative_prompt = "flat, 2d, unrealistic, blurry, low quality"
        
        result = pipe(
            prompt=prompt,
            image=depth_map,
            negative_prompt=negative_prompt,
            num_inference_steps=20,
            controlnet_conditioning_scale=0.7,
            guidance_scale=7.0
        ).images[0]
        
        # Apply mask blending
        result_np = np.array(result.resize((512, 512)))
        face_np = np.array(face_img)
        mask_np = np.array(mask_img)
        mask_3d = np.stack([mask_np/255.0] * 3, axis=-1)
        
        final_result = face_np * (1 - mask_3d) + result_np * mask_3d
        
        return Image.fromarray(final_result.astype(np.uint8))
        
    except Exception as e:
        print(f"Error: {e}")
        return None
    
result = method4_controlnet_depth(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

In [ ]:
# ===================================================================
# METHOD 5: INSTRUCTPIX2PIX HARMONIZATION
# ===================================================================

def method5_instructpix2pix(glasses_img, mask_img, face_img, device="cpu"):
    """
    Method 5: InstructPix2Pix Harmonization
    Uses instruction-based image editing
    """
    
    print("Loading InstructPix2Pix model...")
    
    try:
        from diffusers import StableDiffusionInstructPix2PixPipeline
        
        pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
            "timbrooks/instruct-pix2pix",
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            use_safetensors=True
        )
        
        pipe = pipe.to(device)
        
        # Create composite image
        face_np = np.array(face_img)
        glasses_np = np.array(glasses_img)
        mask_np = np.array(mask_img)
        mask_3d = np.stack([mask_np/255.0] * 3, axis=-1)
        
        composite = face_np * (1 - mask_3d) + glasses_np * mask_3d
        composite_img = Image.fromarray(composite.astype(np.uint8))
        
        # Apply instruction-based editing
        instruction = "Add realistic glasses that blend naturally with the face, adjust lighting and shadows"
        
        result = pipe(
            instruction,
            image=composite_img,
            num_inference_steps=15,
            image_guidance_scale=1.2,
            guidance_scale=7.0
        ).images[0]
        
        return result
        
    except Exception as e:
        print(f"Method 5 failed: {e}")
        return None

result = method5_instructpix2pix(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

In [ ]:
def method6_paint_by_example(glasses_img, mask_img, face_img, device="cuda"):
    """
    Paint by Example Harmonization
    Uses exemplar-based image editing with glasses reference
    """
    from diffusers import PaintByExamplePipeline
    
    pipe = PaintByExamplePipeline.from_pretrained(
        "Fantasy-Studio/Paint-by-Example",
        torch_dtype=torch.float16,
        use_safetensors=True
    ).to(device)
    
    # Prepare images
    face_resized = face_img.resize((512, 512))
    glasses_resized = glasses_img.resize((512, 512))
    mask_resized = mask_img.resize((512, 512))
    
    # Generate harmonized image using glasses as exemplar
    result = pipe(
        image=face_resized,
        mask_image=mask_resized,
        example_image=glasses_resized,
        num_inference_steps=20,
        guidance_scale=5.0
    ).images[0]
    
    return result

result = method6_paint_by_example(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

In [ ]:
def method7_diffusion_style_transfer(glasses_img, mask_img, face_img, device="cuda"):
    """
    Diffusion-Based Style Transfer Harmonization
    Uses img2img diffusion for style harmonization
    """
    from diffusers import StableDiffusionImg2ImgPipeline
    
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16,
        use_safetensors=True
    ).to(device)
    
    # Create composite image
    face_np = np.array(face_img)
    glasses_np = np.array(glasses_img)
    mask_np = np.array(mask_img)
    mask_3d = np.stack([mask_np/255.0] * 3, axis=-1)
    
    composite = face_np * (1 - mask_3d) + glasses_np * mask_3d
    composite_img = Image.fromarray(composite.astype(np.uint8))
    
    # Apply style transfer
    result = pipe(
        prompt="harmonized face with glasses, consistent lighting, natural style, photorealistic",
        image=composite_img,
        negative_prompt="inconsistent, harsh lighting, unrealistic",
        num_inference_steps=25,
        strength=0.3,
        guidance_scale=7.5
    ).images[0]
    
    return result

result = method7_diffusion_style_transfer(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

In [ ]:
def method8_multi_controlnet(glasses_img, mask_img, face_img, device="cuda"):
    """
    Multi-ControlNet Harmonization
    Uses multiple ControlNet conditions simultaneously
    """
    from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, MultiControlNetModel
    from controlnet_aux import CannyDetector, DepthEstimator
    
    # Load multiple ControlNet models
    controlnet_canny = ControlNetModel.from_pretrained(
        "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16
    )
    controlnet_depth = ControlNetModel.from_pretrained(
        "diffusers/controlnet-depth-sdxl-1.0", torch_dtype=torch.float16
    )
    
    # Create multi-ControlNet
    multi_controlnet = MultiControlNetModel([controlnet_canny, controlnet_depth])
    
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        controlnet=multi_controlnet,
        torch_dtype=torch.float16,
        use_safetensors=True
    ).to(device)
    
    # Prepare conditioning images
    canny_detector = CannyDetector()
    depth_estimator = DepthEstimator.from_pretrained("Intel/dpt-hybrid-midas")
    
    canny_image = canny_detector(glasses_img)
    depth_image = depth_estimator(face_img)
    
    # Generate with multiple conditions
    result = pipe(
        prompt="person wearing glasses, perfect lighting, natural harmony, detailed features",
        image=[canny_image, depth_image],
        negative_prompt="blurry, inconsistent, unrealistic",
        num_inference_steps=25,
        controlnet_conditioning_scale=[0.8, 0.6],
        guidance_scale=7.5
    ).images[0]
    
    return result.resize((512, 512))

result = method8_multi_controlnet(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

In [ ]:
def method9_cycle_consistent(glasses_img, mask_img, face_img, device="cuda"):
    """
    Cycle-Consistent Diffusion Harmonization
    Uses multiple passes for cycle consistency
    """
    from diffusers import StableDiffusionImg2ImgPipeline
    
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16,
        use_safetensors=True
    ).to(device)
    
    # Create initial composite
    face_np = np.array(face_img)
    glasses_np = np.array(glasses_img)
    mask_np = np.array(mask_img)
    mask_3d = np.stack([mask_np/255.0] * 3, axis=-1)
    
    composite = face_np * (1 - mask_3d) + glasses_np * mask_3d
    composite_img = Image.fromarray(composite.astype(np.uint8))
    
    # Forward pass
    result1 = pipe(
        prompt="natural face with glasses, consistent lighting, seamless integration",
        image=composite_img,
        num_inference_steps=20,
        strength=0.4,
        guidance_scale=7.0
    ).images[0]
    
    # Refinement pass
    result2 = pipe(
        prompt="photorealistic face with glasses, perfect harmony, professional photography",
        image=result1,
        num_inference_steps=15,
        strength=0.2,
        guidance_scale=6.0
    ).images[0]
    
    return result2

result = method9_cycle_consistent(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

In [ ]:
def method10_latent_diffusion(glasses_img, mask_img, face_img, device="cuda"):
    """
    Latent Diffusion Harmonization
    Uses latent space manipulation for harmonization
    """
    from diffusers import StableDiffusionPipeline
    
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16,
        use_safetensors=True
    ).to(device)
    
    # Generate with latent focus
    result = pipe(
        prompt="high-resolution portrait of person with glasses, perfect latent representation, smooth transitions",
        negative_prompt="blurry, low quality, distorted, unrealistic",
        num_inference_steps=25,
        guidance_scale=7.5,
        width=512,
        height=512
    ).images[0]
    
    # Apply soft mask blending
    result_np = np.array(result)
    face_np = np.array(face_img)
    mask_np = np.array(mask_img)
    
    # Create soft mask
    from PIL import ImageFilter
    soft_mask = Image.fromarray(mask_np).filter(ImageFilter.GaussianBlur(radius=3))
    soft_mask_3d = np.stack([np.array(soft_mask)/255.0] * 3, axis=-1)
    
    final_result = face_np * (1 - soft_mask_3d) + result_np * soft_mask_3d
    return Image.fromarray(final_result.astype(np.uint8))

result = method10_latent_diffusion(src_img, mask_img, bg_img, device=device)
plt.imshow(result)

In [ ]:
def method11_attention_diffusion(glasses_img, mask_img, face_img, device="cuda"):
    """
    Advanced Diffusion with Attention Harmonization
    Uses attention mechanisms for precise harmonization
    """
    from diffusers import StableDiffusionInpaintPipeline
    
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        torch_dtype=torch.float16,
        use_safetensors=True
    ).to(device)
    
    # Prepare attention-focused masks
    face_resized = face_img.resize((512, 512))
    mask_resized = mask_img.resize((512, 512))
    
    # Create gradient mask for attention
    from PIL import ImageFilter
    gradient_mask = Image.fromarray(np.array(mask_resized)).filter(ImageFilter.GaussianBlur(radius=5))
    mask_inverted = Image.fromarray(255 - np.array(mask_resized))
    
    # Generate with attention focus
    result = pipe(
        prompt="photorealistic face with glasses, attention to detail, perfect harmony, natural lighting",
        image=face_resized,
        mask_image=mask_inverted,
        negative_prompt="blurry, low quality, poor attention to detail, inconsistent",
        num_inference_steps=30,
        strength=0.8,
        guidance_scale=8.0
    ).images[0]
    
    # Apply attention-based post-processing
    result_np = np.array(result)
    face_np = np.array(face_resized)
    attention_weight = np.array(gradient_mask) / 255.0
    attention_3d = np.stack([attention_weight] * 3, axis=-1)
    
    final_result = face_np * (1 - attention_3d) + result_np * attention_3d
    return Image.fromarray(final_result.astype(np.uint8))

result = method11_attention_diffusion(src_img, mask_img, bg_img, device=device)
plt.imshow(result)